In [2]:
import pandas as pd
import numpy as np

## Load data

### Questions

In [3]:
questions = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/Kim_VisQA/Questions With Metadata.csv")
questions.head()

,id,vis,vis_type,question,correct,visual,visual_marks,visual_colors,visual_dimensions,visual_position,...,visual_other,compositional,comp_extremum,comp_count,comp_difference,comp_compare,comp_sum,comp_avg,comp_select,comp_multiple
0,1,kong_27,stacked bar,which country's economy will get most worse ov...,Greece,no,no,no,no,no,...,no,yes,yes,no,no,no,no,no,no,no
1,2,kong_27,stacked bar,Which country has the shortest orange bar?,China,yes,yes,yes,yes,no,...,no,yes,yes,no,no,no,no,no,no,no
2,3,kong_27,stacked bar,Which country has the least total percentage o...,Russia,no,no,no,no,no,...,no,yes,no,no,no,no,no,no,no,yes
3,4,kong_27,stacked bar,how many countries in Asia will have their eco...,1,no,no,no,no,no,...,no,yes,no,yes,no,no,no,no,no,no
4,5,kong_27,stacked bar,how many countries' economy will worsen based ...,2,no,no,no,no,no,...,no,yes,no,yes,no,no,no,no,no,no


In [4]:
questions = questions.set_index("id")

In [5]:
len(questions[questions["compositional"] == "no"])

193

### Results

In [6]:
results_1 = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/Kim_VisQA/Results/Claude_VisQA_Complete_1741226669_1.csv").set_index("id")
results_2 = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/Kim_VisQA/Results/Claude_VisQA_Complete_1741226669_2.csv").set_index("id")
results_3 = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/Kim_VisQA/Results/Claude_VisQA_Complete_1741226669_3.csv").set_index("id")

In [7]:
# Merge results
results = results_1.merge(results_2, on='id', suffixes=("", "_2"))
results = results.merge(results_3, on='id', suffixes=("", "_3"))

# Keep track of the first set of results
results["correct_bool_1"] = results["correct_bool"]

# Create consensus correct_bool (True only if all three are True)
# Option 1: Using pandas' built-in operators (recommended)
results["correct_bool"] = (results["correct_bool_1"] & results["correct_bool_2"] & results["correct_bool_3"])

# Option 2: Alternative using np.logical_and (chained)
# results["correct_bool"] = np.where(
#     np.logical_and(
#         np.logical_and(results["correct_bool_1"] == True, results["correct_bool_2"] == True),
#         results["correct_bool_3"] == True
#     ),
#     True,
#     False
# )

# Create consensus results dataframe
results_consensus = results[["correct_bool"]]
# Uncomment to view a specific row
# results.loc[246,:]

In [8]:
results_1_data = pd.read_csv("Results/Clean/Kim_VisQA_data_1707070857_clean.csv").set_index("id")
results_2_data = pd.read_csv("Results/Clean/Kim_VisQA_data_1707405603_clean.csv").set_index("id")
results_3_data = pd.read_csv("Results/Clean/Kim_VisQA_data_1707492757_clean.csv").set_index("id")

In [9]:
results_data = results_1_data.merge(results_2_data, on='id', suffixes=("", "_2"))
results_data = results_data.merge(results_3_data, on='id', suffixes=("", "_3"))
results_data["correct_bool_1"] = results_data["correct_bool"]

results_data["correct_bool"] = np.where(np.logical_and(results_data["correct_bool_1"] == True,
                                                        results_data["correct_bool_2"] == True,
                                                        results_data["correct_bool_3"] == True),
                                        True,
                                        False)
results_data_consensus = results_data[["correct_bool"]]
# results.loc[246,:]

## Results for comparison with original paper

### No data

In [8]:
results_1_meta = questions.join(results_consensus)
results_1_meta.head(3)

,vis,vis_type,question,correct,visual,visual_marks,visual_colors,visual_dimensions,visual_position,visual_axis,...,compositional,comp_extremum,comp_count,comp_difference,comp_compare,comp_sum,comp_avg,comp_select,comp_multiple,correct_bool
id,,,,,,,,,,,,,,,,,,,,,
1,kong_27,stacked bar,which country's economy will get most worse ov...,Greece,no,no,no,no,no,no,...,yes,yes,no,no,no,no,no,no,no,True
2,kong_27,stacked bar,Which country has the shortest orange bar?,China,yes,yes,yes,yes,no,no,...,yes,yes,no,no,no,no,no,no,no,False
3,kong_27,stacked bar,Which country has the least total percentage o...,Russia,no,no,no,no,no,no,...,yes,no,no,no,no,no,no,no,yes,False


In [9]:
## Overall accuracy (%)
sum(results_1_meta["correct_bool"] == True) / len(results_1_meta) * 100

69.63434022257552

In [19]:
results_1_meta_lookup = results_1_meta[results_1_meta["compositional"] == "no"]
print(len(results_1_meta_lookup), "lookup questions")
sum(results_1_meta_lookup["correct_bool"] == True) / len(results_1_meta_lookup) * 100

193 lookup questions


64.76683937823834

In [10]:
results_1_meta_comp = results_1_meta[results_1_meta["compositional"] == "yes"]
print(len(results_1_meta_comp), "compositional questions")
sum(results_1_meta_comp["correct_bool"] == True) / len(results_1_meta_comp) * 100

436 compositional questions


71.78899082568807

In [11]:
results_1_meta_visual = results_1_meta[results_1_meta["visual"] == "yes"]
print(len(results_1_meta_visual), "visual questions")
sum(results_1_meta_visual["correct_bool"] == True) / len(results_1_meta_visual) * 100

76 visual questions


53.94736842105263

In [12]:
results_1_meta_nonvisual = results_1_meta[results_1_meta["visual"] == "no"]
print(len(results_1_meta_nonvisual), "non-visual questions")
sum(results_1_meta_nonvisual["correct_bool"] == True) / len(results_1_meta_nonvisual) * 100

553 non-visual questions


71.79023508137432

In [13]:
results_1_meta_visual_lookup = results_1_meta[np.logical_and(results_1_meta["visual"] == "yes", results_1_meta["compositional"] == "no")]
print(len(results_1_meta_visual_lookup), "visual lookup questions")
sum(results_1_meta_visual_lookup["correct_bool"] == True) / len(results_1_meta_visual_lookup) * 100

52 visual lookup questions


51.92307692307693

In [14]:
results_1_meta_visual_comp = results_1_meta[np.logical_and(results_1_meta["visual"] == "yes", results_1_meta["compositional"] == "yes")]
print(len(results_1_meta_visual_comp), "visual compositional questions")
sum(results_1_meta_visual_comp["correct_bool"] == True) / len(results_1_meta_visual_comp) * 100

24 visual compositional questions


58.333333333333336

In [15]:
results_1_meta_nonvisual_lookup = results_1_meta[np.logical_and(results_1_meta["visual"] == "no", results_1_meta["compositional"] == "no")]
print(len(results_1_meta_nonvisual_lookup), "non-visual lookup questions")
sum(results_1_meta_nonvisual_lookup["correct_bool"] == True) / len(results_1_meta_nonvisual_lookup) * 100

141 non-visual lookup questions


69.50354609929079

In [16]:
results_1_meta_nonvisual_comp = results_1_meta[np.logical_and(results_1_meta["visual"] == "no", results_1_meta["compositional"] == "yes")]
print(len(results_1_meta_nonvisual_comp), "non-visual compositional questions")
sum(results_1_meta_nonvisual_comp["correct_bool"] == True) / len(results_1_meta_nonvisual_comp) * 100

412 non-visual compositional questions


72.57281553398059

### Data

### Deeper Dive: Visual Lookup w/ Data

In [33]:
results_1_data_meta.loc[314,:]

vis                                                      stacked_bar_h
vis_type                                                   stacked bar
question             What percentage of the yield in No 475 came fr...
correct                                                       9.165646
visual                                                              no
visual_marks                                                        no
visual_colors                                                       no
visual_dimensions                                                   no
visual_position                                                     no
visual_axis                                                         no
visual_other                                                        no
compositional                                                      yes
comp_extremum                                                       no
comp_count                                                          no
comp_d

In [34]:
results_1_data_meta_visual_lookup[results_1_data_meta_visual_lookup["correct_bool"] == False]

,vis,vis_type,question,correct,visual,visual_marks,visual_colors,visual_dimensions,visual_position,visual_axis,...,compositional,comp_extremum,comp_count,comp_difference,comp_compare,comp_sum,comp_avg,comp_select,comp_multiple,correct_bool
id,,,,,,,,,,,,,,,,,,,,,
85,kong_5,stacked bar,What is the percentage of red Protestants?,42,yes,no,yes,no,no,no,...,no,no,no,no,no,no,no,no,no,False
293,stacked_bar_h,stacked bar,What is the value of the orange Bar in Wiscons...,60.93333,yes,yes,yes,no,no,no,...,no,no,no,no,no,no,no,no,no,False
300,stacked_bar_h,stacked bar,what is the value of blue in manchuria?,72.9,yes,no,yes,no,no,no,...,no,no,no,no,no,no,no,no,no,False
303,stacked_bar_h,stacked bar,What does the red field represent?,Grand Rapids,yes,no,yes,no,no,no,...,no,no,no,no,no,no,no,no,no,False
307,stacked_bar_h,stacked bar,what is the value of yellow on velvet?,87.63333,yes,no,yes,no,no,no,...,no,no,no,no,no,no,no,no,no,False
315,stacked_bar_h,stacked bar,What's the red value for Giabron?,43.56666,yes,no,yes,no,no,no,...,no,no,no,no,no,no,no,no,no,False
331,stacked_bar_h,stacked bar,What is the value of the red bar in No 462?,44.83334,yes,yes,yes,no,no,no,...,no,no,no,no,no,no,no,no,no,False
339,stacked_bar_h,stacked bar,What site does green represent?,University Farm,yes,no,yes,no,no,no,...,no,no,no,no,no,no,no,no,no,False
340,stacked_bar_h,stacked bar,What does the green bar represent?,University Farm,yes,yes,yes,no,no,no,...,no,no,no,no,no,no,no,no,no,False


In [35]:
results_1_data.loc[339, :].response

'In the image provided, the color green represents the site "Morris" according to the legend on the right side of the chart.'

In [36]:
results_2_data.loc[339, :].response

'In the provided bar chart, the green color represents the site "Morris." Each color in the chart corresponds to a different site where barley yields for various varieties were measured, and the legend on the right side of the chart indicates which color is associated with each site.'

In [37]:
results_3_data.loc[339, :].response

'In the provided image, the green color represents the site "Morris."'

### Deeper Dive: Tasks (without data)

In [20]:
results_1_data_meta_extrema = results_1_meta[results_1_meta["comp_extremum"] == "yes"]
print(len(results_1_data_meta_extrema))
np.mean(results_1_data_meta_extrema["correct_bool"]) * 100

167


79.64071856287424

In [21]:
results_1_data_meta_compare = results_1_meta[results_1_meta["comp_compare"] == "yes"]
print(len(results_1_data_meta_compare))
np.mean(results_1_data_meta_compare["correct_bool"]) * 100

25


72.0

In [10]:
results_1_data_meta_derived = results_1_meta[
    np.logical_or(
        np.logical_or(
            results_1_meta["comp_difference"] == "yes",
            results_1_meta["comp_sum"] == "yes"
        ),
        np.logical_or(
            results_1_meta["comp_avg"] == "yes",
            results_1_meta["comp_count"] == "yes"
        )
    )
]

print(len(results_1_data_meta_derived))
np.mean(results_1_data_meta_derived["correct_bool"]) * 100

153


62.091503267973856

In [11]:
results_1_data_meta_multiple = results_1_meta[results_1_meta["comp_multiple"] == "yes"]
print(len(results_1_data_meta_multiple))
np.mean(results_1_data_meta_multiple["correct_bool"]) * 100

70


74.28571428571429

In [12]:
results_1_data_meta_multiple[results_1_data_meta_multiple["correct_bool"] == True].head(1)

,vis,vis_type,question,correct,visual,visual_marks,visual_colors,visual_dimensions,visual_position,visual_axis,...,compositional,comp_extremum,comp_count,comp_difference,comp_compare,comp_sum,comp_avg,comp_select,comp_multiple,correct_bool
id,,,,,,,,,,,,,,,,,,,,,
27,kong_30,stacked bar,Which country's has the lowest summed value fr...,China,no,no,no,no,no,no,...,yes,no,no,no,no,no,no,no,yes,True


In [54]:
results_1_meta_axis = results_1_data_meta[results_1_meta["visual_axis"] == "yes"]
np.mean(results_1_meta_axis["correct_bool"])

1.0